# RAG-система с ChromaDB + BM25 Hybrid Search

## Стек
- **Vector DB:** ChromaDB (HNSW, локально)
- **Sparse search:** rank_bm25 (BM25Okapi)
- **Hybrid fusion:** Reciprocal Rank Fusion (RRF)
- **Embeddings:** `intfloat/multilingual-e5-large` (1024d)
- **LLM:** OpenAI gpt-4.1-mini

## Что сравниваем
1. **Dense only** — ChromaDB HNSW
2. **Sparse only** — BM25
3. **Hybrid** — RRF(Dense + Sparse)


In [2]:
%pip install -q chromadb rank-bm25 sentence-transformers langchain langchain-community langchain-openai "openai>=1.99.1,<2.25.0" nltk pandas python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Импорты и конфигурация

In [1]:
import os
import time
from pathlib import Path
from typing import List, Dict

from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv(override=True)

DOCS_DIR        = Path("./docs")
COLLECTION_NAME = "hw6_docs"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
OPENAI_MODEL    = "gpt-4.1-mini"
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 150
TOP_K           = 5
RRF_K           = 60

print("✅ Импорты загружены")
print(f"   Docs dir : {DOCS_DIR.resolve()}")
print(f"   Embedding: {EMBEDDING_MODEL}")
print(f"   LLM      : {OPENAI_MODEL}")
print(f"   API key  : {'✅ найден' if os.getenv('OPENAI_API_KEY') else '❌ не найден'}")

✅ Импорты загружены
   Docs dir : /Users/kornilovyv/Desktop/learning/otus/llm_dd/hw_06/docs
   Embedding: intfloat/multilingual-e5-large
   LLM      : gpt-4.1-mini
   API key  : ✅ найден


## Загрузка и чанкинг документов

In [2]:
import re

def clean_doc_text(text: str) -> str:
    """
    Очищает Yandex Docs template-синтаксис из MD-файлов:
    - {{ variable }} → подставляем читаемые имена или убираем
    - {% if %}...{% endif %} → убираем теги, оставляем содержимое
    - {% note %}...{% endnote %} → убираем теги, оставляем содержимое
    - {% cut %}...{% endcut %} → убираем теги, оставляем содержимое
    - {% include ... %} → убираем полностью
    - {#T} → убираем
    """
    # Заменяем известные переменные на читаемые значения
    replacements = {
        "{{ datalens-full-name }}": "DataLens",
        "{{ datalens-short-name }}": "DataLens",
        "{{ datalens-name }}": "DataLens",
        "{{ link-datalens-main }}": "https://datalens.ru",
    }
    for template, value in replacements.items():
        text = text.replace(template, value)

    # Убираем оставшиеся {{ ... }}
    text = re.sub(r"\{\{[^}]+\}\}", "", text)

    # Убираем {% include ... %} полностью
    text = re.sub(r"\{%\s*include[^%]+%\}", "", text)

    # Убираем теги {% if/elif/else/endif %}, оставляем содержимое между ними
    text = re.sub(r"\{%[^%]+%\}", "", text)

    # Убираем {#T}
    text = re.sub(r"\{#T\}", "", text)

    # Убираем синтаксис таблиц Yandex Docs (#| и ||)
    text = re.sub(r"^#\|$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\|\|", "|", text, flags=re.MULTILINE)

    # Убираем image-теги вида ![image](path =700x495)
    text = re.sub(r"!\[.*?\]\(.*?\)", "", text)

    # Убираем строки с только пробелами/пустые дубли
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_metadata(source_path: str) -> dict:
    """Извлекает метаданные из пути файла."""
    path = Path(source_path)
    parts = path.parts

    # Определяем category (concepts / operations / другое) и section (chart / dataset / ...)
    category = "general"
    section = "general"
    for i, part in enumerate(parts):
        if part in ("concepts", "operations"):
            category = part
            if i + 1 < len(parts):
                section = parts[i + 1]
            break

    return {
        "filename": path.stem,
        "category": category,   # concepts | operations
        "section": section,     # chart | dataset | ...
    }


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
raw_docs = loader.load()
print(f"📄 Загружено документов: {len(raw_docs)}")

# Очищаем текст и добавляем метаданные
for doc in raw_docs:
    doc.page_content = clean_doc_text(doc.page_content)
    doc.metadata.update(extract_metadata(doc.metadata["source"]))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)
chunks = splitter.split_documents(raw_docs)
print(f"✂️  Чанков после сплиттинга: {len(chunks)}")

# Назначаем уникальный id каждому чанку
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"chunk_{i:04d}"

print("\n--- Пример чанка (после очистки) ---")
print(chunks[0].page_content[:400])
print("\nMetadata:", chunks[0].metadata)

📄 Загружено документов: 15
✂️  Чанков после сплиттинга: 104

--- Пример чанка (после очистки) ---
---
title: "Настройки чарта в DataLens"
description: "В этой статье вы узнаете о настройках чарта в DataLens, а также узнаете, как отменить и восстановить изменения в чартах."
---

# Настройки чарта в DataLens

Вы можете настраивать чарты. Например, добавить отображение легенды, настроить цветовую схему, указать собственный заголовок.

Доступность настроек зависит от типа настраиваемого чарта.

Metadata: {'source': 'docs/concepts/chart/settings.md', 'filename': 'settings', 'category': 'concepts', 'section': 'chart', 'chunk_id': 'chunk_0000'}


## ChromaDB: создание коллекции и индексация

ChromaDB использует **HNSW** через `hnswlib`. Параметры конфигурируются при создании коллекции:

| Параметр | Значение | Влияние |
|---|---|---|
| `hnsw:M` | 16 | Число рёбер на узел. Больше → выше recall, больше RAM |
| `hnsw:ef_construction` | 100 | Глубина поиска при построении. Больше → лучше граф, медленнее индексация |
| `hnsw:ef_search` | 50 | Глубина поиска при запросе. Trade-off: скорость vs recall |
| `hnsw:space` | cosine | Метрика расстояния |

In [4]:
from sentence_transformers import SentenceTransformer
from chromadb import EmbeddingFunction, Documents, Embeddings, PersistentClient

class E5EmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_name: str, device: str = "cpu"):
        self.model = SentenceTransformer(model_name, device=device)

    def __call__(self, input: Documents) -> Embeddings:
        # multilingual-e5 обучена с двумя префиксами:
        #   "query:"   — для поисковых запросов (короткие тексты)
        #   "passage:" — для индексируемых документов (длинные тексты)
        # Модель оптимизирует косинусное расстояние между query и passage,
        # поэтому важно использовать правильный префикс для каждого типа.
        # Но ChromaDB's EmbeddingFunction не предполагает такого разделения, поэтому используем эвристику.
        # Длина < 200 символов: запросы обычно короткие,
        # чанки (800 символов по конфигу) всегда длиннее порога.
        texts = [
            f"query: {t}" if len(t) < 200 else f"passage: {t}"
            for t in input
        ]
        return self.model.encode(texts, normalize_embeddings=True).tolist()


print(f"⏳ Загрузка модели {EMBEDDING_MODEL} на MPS...")
t0 = time.time()
ef = E5EmbeddingFunction(EMBEDDING_MODEL, device="mps")
test_vec = ef(["тестовый запрос"])
print(f"✅ Модель загружена за {time.time()-t0:.1f}с. Размерность: {len(test_vec[0])}d")

chroma_client = PersistentClient(path="./chroma_db")

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"🗑️  Удалена старая коллекция '{COLLECTION_NAME}'")
except Exception:
    pass

HNSW_CONFIG = {
    "space": "cosine",
    "max_neighbors": 16,
    "ef_construction": 100,
    "ef_search": 50,
}

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=ef,
    configuration={"hnsw": HNSW_CONFIG}
)
print(f"✅ Коллекция '{COLLECTION_NAME}' создана")
print(f"   HNSW {HNSW_CONFIG}")

print("\n⏳ Индексация чанков...")
t0 = time.time()
collection.add(
    ids=[c.metadata["chunk_id"] for c in chunks],
    documents=[c.page_content for c in chunks],
    metadatas=[c.metadata for c in chunks]
)
print(f"🎉 Проиндексировано {len(chunks)} чанков за {time.time()-t0:.1f}с")

⏳ Загрузка модели intfloat/multilingual-e5-large на MPS...
✅ Модель загружена за 6.3с. Размерность: 1024d
🗑️  Удалена старая коллекция 'hw6_docs'
✅ Коллекция 'hw6_docs' создана
   HNSW {'space': 'cosine', 'max_neighbors': 16, 'ef_construction': 100, 'ef_search': 50}

⏳ Индексация чанков...
🎉 Проиндексировано 104 чанков за 5.8с


## BM25: построение индекса

**BM25Okapi** — улучшенная версия BM25 с нормализацией частоты термина:

```
score(q, d) = Σ IDF(qi) * (tf(qi,d) * (k1+1)) / (tf(qi,d) + k1*(1-b+b*|d|/avgdl))
```

- `k1` (default 1.5) — насыщение TF: при больших значениях частота слова влияет сильнее
- `b` (default 0.75) — нормализация длины: 0 = без нормализации, 1 = полная

In [5]:
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from rank_bm25 import BM25Okapi

try:
    _stop_ru = set(stopwords.words('russian'))
    _stop_en = set(stopwords.words('english'))
    STOP_WORDS = _stop_ru | _stop_en
except Exception:
    STOP_WORDS = set()

def tokenize(text: str) -> List[str]:
    """Токенизация с удалением стоп-слов и пунктуации."""
    try:
        tokens = word_tokenize(text.lower())
    except Exception:
        tokens = text.lower().split()
    return [t for t in tokens if t.isalpha() and t not in STOP_WORDS]

print("⏳ Построение BM25 индекса...")
t0 = time.time()

corpus_texts = [c.page_content for c in chunks]
tokenized_corpus = [tokenize(text) for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print(f"✅ BM25 индекс готов за {time.time()-t0:.2f}с")
print(f"   Документов: {len(corpus_texts)}")
print(f"   Параметры: k1=1.5, b=0.75")

sample_tokens = tokenize("Как настроить отображение пустых значений null в чарте?")
print(f"\n🔬 Пример токенизации: {sample_tokens}")

⏳ Построение BM25 индекса...
✅ BM25 индекс готов за 0.04с
   Документов: 104
   Параметры: k1=1.5, b=0.75

🔬 Пример токенизации: ['настроить', 'отображение', 'пустых', 'значений', 'null', 'чарте']


## Функции поиска: Dense / Sparse / Hybrid

In [6]:
def search_dense(query: str, k: int = TOP_K) -> List[Dict]:
    """ChromaDB HNSW — dense vector search."""
    results = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    return [
        {
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            # ChromaDB distance: cosine distance (0=identical, 2=opposite)
            # переводим в score: чем меньше distance, тем выше score
            "score": 1 - results["distances"][0][i],
        }
        for i in range(len(results["ids"][0]))
    ]


def search_bm25(query: str, k: int = TOP_K) -> List[Dict]:
    """BM25 sparse search."""
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)

    # Берём топ-k по score
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {
            "chunk_id": chunks[i].metadata["chunk_id"],
            "text": corpus_texts[i],
            "metadata": chunks[i].metadata,
            "score": float(scores[i]),
        }
        for i in top_indices
        if scores[i] > 0  # отбрасываем нулевые совпадения
    ]


def reciprocal_rank_fusion(
    dense_results: List[Dict],
    sparse_results: List[Dict],
    k: int = RRF_K
) -> List[Dict]:
    """
    Reciprocal Rank Fusion:
    RRF_score(d) = Σ 1 / (k + rank(d))

    Объединяет ранжирование из двух списков без нормализации исходных скоров.
    k=60 — эмпирически хорошее значение из оригинальной статьи Cormack et al. 2009.
    """
    rrf_scores: Dict[str, float] = {}
    all_docs: Dict[str, Dict] = {}

    for rank, doc in enumerate(dense_results, start=1):
        cid = doc["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + rank)
        all_docs[cid] = doc

    for rank, doc in enumerate(sparse_results, start=1):
        cid = doc["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + rank)
        all_docs[cid] = doc

    sorted_ids = sorted(rrf_scores, key=lambda cid: rrf_scores[cid], reverse=True)
    return [
        {**all_docs[cid], "score": rrf_scores[cid]}
        for cid in sorted_ids
    ]


def search_hybrid(query: str, k: int = TOP_K) -> List[Dict]:
    """Hybrid search: RRF(Dense + BM25)."""
    dense  = search_dense(query, k=k)
    sparse = search_bm25(query,  k=k)
    fused  = reciprocal_rank_fusion(dense, sparse)
    return fused[:k]

## Сравнение: Dense vs Sparse vs Hybrid

In [7]:
def print_results(title: str, results: List[Dict], top_n: int = 3):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for i, r in enumerate(results[:top_n], 1):
        fname    = r["metadata"].get("filename", "?")
        category = r["metadata"].get("category", "?")
        print(f"[{i}] score={r['score']:.4f} | {category}/{fname}")
        print(f"    {r['text'][:200].strip()}...")


# Запрос 1: точное совпадение слов "null" → BM25 должен выиграть
# Запрос 2: семантика без точных слов → Dense должен выиграть
test_queries = [
    "как настроить null значения на диаграмме",   # точные термины → BM25
    "визуализация пробелов в данных",              # семантика → Dense
]

for query in test_queries:
    print(f"\n🔎 Запрос: '{query}'")
    print_results("Dense (HNSW)",  search_dense(query))
    print_results("Sparse (BM25)", search_bm25(query))
    print_results("Hybrid (RRF)",  search_hybrid(query))


🔎 Запрос: 'как настроить null значения на диаграмме'

  Dense (HNSW)
[1] score=0.8814 | operations/chart-null-settings
    ---
title: "Как настроить отображение пустых (null) значений в чарте DataLens"
description: "Следуя данной инструкции, вы сможете настроить отображение пустых (null) значений в чарте DataLens."
---

#...
[2] score=0.8811 | operations/chart-null-settings
    Вы можете указать, как будут отображаться пустые значения на диаграмме, в настройках секции чарта:

1. В секции с показателем, отображение значения которого надо настроить, в правом верхнем углу нажми...
[3] score=0.8803 | operations/chart-null-settings
    1. Нажмите кнопку **Применить**.

Если в исходных данных совсем нет строки, опция **Пустые значения (null)** не изменит отображение показателя на диаграмме. Например, если в источнике нет строки с опр...

  Sparse (BM25)
[1] score=10.5344 | operations/chart-null-settings
    Вы можете указать, как будут отображаться пустые значения на диаграмме, в настройках

### Анализ результатов

**Запрос 1: `"как настроить null значения на диаграмме"`**

Все три метода нашли правильный документ (`operations/chart-null-settings`). Запрос содержит точные термины из документа, поэтому даже Dense справился — слова "null", "настроить", "диаграмме" есть в тексте. Разницы нет, тест не выявил преимущество BM25.

**Запрос 2: `"визуализация пробелов в данных"`**

| Метод | Топ-1 | Топ-2 | Топ-3 |
|---|---|---|---|
| Dense | concepts/settings | concepts/settings | concepts/index |
| BM25 | concepts/index | operations/create-sql-chart | operations/create-sql-chart |
| Hybrid | concepts/settings | concepts/index | concepts/settings |

BM25 нашёл `create-sql-chart` — там встречается слово "визуализация", но по смыслу нерелевантно. Dense лучше уловил семантику: "пробелы в данных" ≈ null values, хотя слова разные. Hybrid правильно поднял `concepts/settings` на первое место и добавил `concepts/index` от BM25.

**Вывод**

Задуманный сценарий "BM25 выигрывает на точных терминах" не подтвердился — запрос 1 слишком лёгкий для обоих методов. Чтобы честно показать слабость Dense, нужен запрос с редким термином.

## RAG Pipeline

In [8]:
import httpx
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


_http_client = httpx.Client(verify=False, timeout=60.0)

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0.1,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    http_client=_http_client,
)

RAG_PROMPT = ChatPromptTemplate.from_template("""
Ты — ассистент по документации Yandex DataLens.
Ответь на вопрос пользователя, используя ТОЛЬКО предоставленный контекст.
Если в контексте нет ответа, скажи: "В документации нет информации об этом".
Не придумывай факты.

Контекст:
{context}

Вопрос: {question}

Ответ:
""")

rag_chain = RAG_PROMPT | llm | StrOutputParser()


def ask(question: str, search_fn=search_hybrid) -> str:
    """Задать вопрос RAG-системе."""
    docs = search_fn(question)
    context = "\n\n".join(
        f"[{r['metadata'].get('category','?')}/{r['metadata'].get('filename','?')}]\n{r['text']}"
        for r in docs
    )
    return rag_chain.invoke({"context": context, "question": question})

In [9]:
question = "Как настроить отображение пустых (null) значений в чарте?"

print(f"❓ Вопрос: {question}\n")
answer = ask(question)
display(Markdown(answer))

❓ Вопрос: Как настроить отображение пустых (null) значений в чарте?



Настроить отображение пустых (null) значений в чарте DataLens можно в настройках секции чарта, где отображается соответствующий показатель:

1. В секции с показателем, для которого нужно настроить отображение пустых значений, наведите указатель мыши на правый верхний угол секции и нажмите появившийся значок.
2. В опции **Пустые значения (null)** выберите один из вариантов отображения:
   - **Не отображать** — пустые значения не показываются, на диаграмме это будет разрыв линии, пропуск столбца или точки.
   - **Соединять** — соседние точки с непустыми значениями соединяются линией, пропуская null.
   - **Отображать как 0** — пустые значения отображаются как нули.
   - **Использовать предыдущее** — пустые значения заменяются значением предыдущей точки (доступно для накопительной диаграммы на оси Y).
3. Нажмите кнопку **Применить** для сохранения настроек.

Важно: если в исходных данных отсутствует строка с определённой датой или категорией, пустые значения для неё не отобразятся, даже если выбрана опция отображения null. Чтобы отобразить нулевое значение для отсутствующей строки, её нужно добавить в источник с значением `null`.

In [10]:
# Тест на галлюцинации — вопрос вне документации
fake_q = "Сколько стоит подписка на DataLens?"

print(f"❓ Вопрос (нет в документах): {fake_q}\n")
answer_fake = ask(fake_q)
print(answer_fake)

❓ Вопрос (нет в документах): Сколько стоит подписка на DataLens?

В документации нет информации об этом.


## Бенчмарк скорости поиска

In [11]:
import pandas as pd
import statistics

BENCH_QUERIES = [
    "как создать чарт",
    "настройка null значений",
    "добавить описание к чарту",
    "мультидатасетный чарт",
    "замена датасета в чарте",
]

def bench(name: str, fn, queries: List[str], runs: int = 5):
    times = []
    for q in queries:
        for _ in range(runs):
            t0 = time.perf_counter()
            fn(q)
            times.append((time.perf_counter() - t0) * 1000)  # ms
    return {
        "method": name,
        "mean_ms": round(statistics.mean(times), 2),
        "median_ms": round(statistics.median(times), 2),
        "p95_ms": round(sorted(times)[int(0.95 * len(times))], 2),
    }

results = [
    bench("Dense (HNSW)", search_dense, BENCH_QUERIES),
    bench("Sparse (BM25)", search_bm25, BENCH_QUERIES),
    bench("Hybrid (RRF)", search_hybrid, BENCH_QUERIES),
]

df = pd.DataFrame(results)
print("\n📊 Результаты бенчмарка (меньше = быстрее):")
display(df)


📊 Результаты бенчмарка (меньше = быстрее):


,method,mean_ms,median_ms,p95_ms
0,Dense (HNSW),27.81,20.30,58.26
1,Sparse (BM25),0.06,0.05,0.07
2,Hybrid (RRF),22.13,21.65,23.57


BM25 ~0.06ms — это математика в RAM, не зависит от прогрева.

Hybrid (22.13ms mean) быстрее Dense (27.81ms mean), хотя выполняет оба поиска. Объяснение: Hybrid = Dense + BM25, но BM25 добавляет лишь 0.06ms, а само ядро ChromaDB уже прогрето и выдаёт стабильные ~20ms без выбросов.

В cold-start сценарии Dense непредсказуем. Hybrid даёт лучшее качество поиска за счёт объединения двух методов и при этом сопоставим по скорости с Dense.

## Изучение влияния параметров HNSW

ChromaDB не позволяет менять `ef_search` после создания коллекции через API,
но можно создать несколько коллекций с разными параметрами и сравнить recall.

In [12]:
def make_collection(name: str, max_neighbors: int, ef_construction: int, ef_search: int):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    col = chroma_client.create_collection(
        name=name,
        embedding_function=ef,
        configuration={
            "hnsw": {
                "space": "cosine",
                "max_neighbors": max_neighbors,
                "ef_construction": ef_construction,
                "ef_search": ef_search,
            }
        }
    )
    col.add(
        ids=[c.metadata["chunk_id"] for c in chunks],
        documents=[c.page_content for c in chunks],
        metadatas=[c.metadata for c in chunks]
    )
    return col


configs = [
    {"name": "hnsw_fast", "max_neighbors": 8, "ef_construction": 50, "ef_search": 10},
    {"name": "hnsw_default", "max_neighbors": 16, "ef_construction": 100, "ef_search": 50},
    {"name": "hnsw_accurate", "max_neighbors": 32, "ef_construction": 200, "ef_search": 200},
]

print("⏳ Создание коллекций с разными параметрами HNSW...")
collections = {}
build_times = {}
for cfg in configs:
    t0 = time.time()
    collections[cfg["name"]] = make_collection(**cfg)
    build_times[cfg["name"]] = round(time.time() - t0, 2)
    print(f"  ✅ {cfg['name']}: max_neighbors={cfg['max_neighbors']}, ef_c={cfg['ef_construction']}, ef_s={cfg['ef_search']} — {build_times[cfg['name']]}с")

⏳ Создание коллекций с разными параметрами HNSW...
  ✅ hnsw_fast: max_neighbors=8, ef_c=50, ef_s=10 — 5.72с
  ✅ hnsw_default: max_neighbors=16, ef_c=100, ef_s=50 — 5.54с
  ✅ hnsw_accurate: max_neighbors=32, ef_c=200, ef_s=200 — 5.52с


In [14]:
def query_collection(col, query: str, k: int = TOP_K) -> List[str]:
    """Возвращает список chunk_id для запроса."""
    r = col.query(query_texts=[query], n_results=k, include=[])
    return r["ids"][0]


def recall_at_k(retrieved: List[str], relevant: List[str]) -> float:
    """Recall@k = |retrieved ∩ relevant| / |relevant|"""
    if not relevant:
        return 0.0
    return len(set(retrieved) & set(relevant)) / len(relevant)


# Accurate конфиг — ground truth
gt_col = collections["hnsw_accurate"]

rows = []
for query in BENCH_QUERIES:
    gt_ids = query_collection(gt_col, query, k=TOP_K)
    for cfg in configs:
        col = collections[cfg["name"]]
        t0 = time.perf_counter()
        ids = query_collection(col, query, k=TOP_K)
        latency_ms = (time.perf_counter() - t0) * 1000
        rows.append({
            "config": cfg["name"],
            "max_neighbors": cfg["max_neighbors"],
            "ef_search": cfg["ef_search"],
            "query": query,
            "recall@5": recall_at_k(ids, gt_ids),
            "latency_ms": round(latency_ms, 2),
        })

df_hnsw = pd.DataFrame(rows)
summary = df_hnsw.groupby(["config", "max_neighbors", "ef_search"]).agg(
    recall_mean=("recall@5", "mean"),
    latency_mean=("latency_ms", "mean")
).round(3).reset_index()

print("\n📊 HNSW: Trade-off между скоростью и recall (vs accurate конфиг):")
display(summary)


📊 HNSW: Trade-off между скоростью и recall (vs accurate конфиг):


,config,max_neighbors,ef_search,recall_mean,latency_mean
0,hnsw_accurate,32,200,1.00,20.434
1,hnsw_default,16,50,1.00,20.150
2,hnsw_fast,8,10,0.96,20.242


На 104 документах параметры HNSW не влияют на latency вообще — три конфига с разбросом ef_search в 20 раз (10 vs 200) дают одинаковые ~20ms. Единственное наблюдаемое отличие — recall у hnsw_fast ниже на 4%. Для production-выводов о trade-off скорость/качество нужен корпус на порядок больше.

## Итоги

### ANN алгоритм ChromaDB — HNSW

- **Структура:** многоуровневый граф, навигация от разреженного верхнего уровня к плотному нижнему
- **Trade-off:** `max_neighbors` и `ef_search` управляют компромиссом скорость ↔ recall
- **Инкрементальные обновления:** поддерживаются без перестроения всего индекса

### Dense vs Sparse vs Hybrid

| Метод | Сильные стороны | Слабые стороны |
|---|---|---|
| Dense (HNSW) | семантика, синонимы | не ловит точные термины |
| Sparse (BM25) | точные слова, коды, имена | нет семантики |
| Hybrid (RRF) | лучшее из обоих | чуть выше latency |

### RRF формула

```
RRF_score(d) = 1/(60 + rank_dense(d)) + 1/(60 + rank_sparse(d))
```

Работает на рангах, а не на скорах — нет проблемы с нормализацией разных шкал.


## Сравнение similarity metrics: cosine vs euclidean vs dot product

In [16]:
print("⏳ Создание коллекций с разными метриками...")

metric_collections = {}
for space in ["cosine", "l2", "ip"]:
    name = f"metric_{space}"
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    col = chroma_client.create_collection(
        name=name,
        embedding_function=ef,
        configuration={"hnsw": {"space": space, "max_neighbors": 16, "ef_construction": 100, "ef_search": 50}}
    )
    col.add(
        ids=[c.metadata["chunk_id"] for c in chunks],
        documents=[c.page_content for c in chunks],
        metadatas=[c.metadata for c in chunks]
    )
    metric_collections[space] = col
    print(f"  ✅ {space}")

METRIC_QUERIES = [
    "как создать чарт",
    "настройка null значений",
    "мультидатасетный чарт",
]

rows = []
for query in METRIC_QUERIES:
    # ground truth — топ-5 id из cosine (наша основная метрика)
    gt = metric_collections["cosine"].query(
        query_texts=[query], n_results=TOP_K, include=[]
    )["ids"][0]

    for space, col in metric_collections.items():
        r = col.query(query_texts=[query], n_results=TOP_K, include=["metadatas"])
        ids = r["ids"][0]
        top1_file = r["metadatas"][0][0].get("filename", "?")
        rows.append({
            "query": query,
            "space": space,
            "top1_file": top1_file,
            "recall@5_vs_cosine": round(len(set(ids) & set(gt)) / len(gt), 2),
        })

df_metrics = pd.DataFrame(rows)
# Сводная таблица: recall по каждой метрике
pivot = df_metrics.pivot_table(
    index="space", columns="query", values="recall@5_vs_cosine"
).round(2)

print("\n📊 Recall@5 относительно cosine (1.0 = идентичный порядок):")
display(pivot)

print("\n📋 Топ-1 документ по каждой метрике:")
display(df_metrics.pivot_table(
    index="space", columns="query", values="top1_file", aggfunc="first"
))

⏳ Создание коллекций с разными метриками...
  ✅ cosine
  ✅ l2
  ✅ ip

📊 Recall@5 относительно cosine (1.0 = идентичный порядок):


query,как создать чарт,мультидатасетный чарт,настройка null значений
space,,,
cosine,1.0,1.0,1.0
ip,1.0,1.0,1.0
l2,1.0,1.0,1.0



📋 Топ-1 документ по каждой метрике:


query,как создать чарт,мультидатасетный чарт,настройка null значений
space,,,
cosine,create-chart,multidataset-chart,chart-null-settings
ip,create-chart,multidataset-chart,chart-null-settings
l2,create-chart,multidataset-chart,chart-null-settings



| Метрика | Что меряет | Когда использовать |
|---|---|---|
| `cosine` | угол между векторами | нормализованные эмбеддинги (наш случай) |
| `l2` (euclidean) | расстояние в пространстве | ненормализованные векторы |
| `ip` (dot product) | скалярное произведение | когда длина вектора несёт смысл |

### Почему все три метрики дали одинаковый результат?

`multilingual-e5-large` генерирует **нормализованные** векторы (`normalize_embeddings=True`, ‖v‖ = 1).

На единичных векторах метрики математически связаны:

```
cosine(u,v) = u·v / (‖u‖·‖v‖) = u·v = ip(u,v)     # cosine ≡ ip на нормализованных
l2²(u,v)   = ‖u-v‖² = ‖u‖² - 2u·v + ‖v‖² = 2 - 2·cosine(u,v)
```

Поэтому `cosine` и `ip` дают **идентичный** порядок результатов. `l2` монотонно зависит от `cosine` (меньше l2 = больше cosine), поэтому ранжирование тоже **совпадает**.

**Вывод:** при работе с нормализованными эмбеддингами выбор метрики не влияет на качество поиска. Можно использовать `cosine` как наиболее интуитивную (0 = одинаковые, 1 = ортогональные, -1 = противоположные).

**Когда метрики расходились бы:** при ненормализованных векторах, где длина несёт информацию (например, TF-IDF или Word2Vec без нормализации).

## Фильтрация по метаданным

ChromaDB поддерживает фильтрацию через `where=` — условие применяется **до** векторного поиска,
то есть HNSW ищет только среди отфильтрованных документов.

Доступные поля в наших метаданных: `category` (concepts / operations), `section` (chart), `filename`.

In [17]:
def search_dense_filtered(query: str, filters: dict, k: int = TOP_K) -> List[Dict]:
    """Dense поиск с фильтрацией по метаданным через ChromaDB where=."""
    # ChromaDB синтаксис: {"field": {"$eq": value}} или {"$and": [...]}
    where = {key: {"$eq": val} for key, val in filters.items()}
    if len(where) > 1:
        where = {"$and": [{k: v} for k, v in where.items()]}
    else:
        where = next(iter(where.items()))
        where = {where[0]: where[1]}

    results = collection.query(
        query_texts=[query],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"]
    )
    return [
        {
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "score": 1 - results["distances"][0][i],
        }
        for i in range(len(results["ids"][0]))
    ]


query = "как создать чарт"

print(f"🔎 Запрос: '{query}'\n")

# Без фильтра
no_filter = search_dense(query)
print("--- Без фильтра ---")
for r in no_filter[:3]:
    print(f"  score={r['score']:.4f} | {r['metadata']['category']}/{r['metadata']['filename']}")

# Только operations (пошаговые инструкции)
ops_only = search_dense_filtered(query, {"category": "operations"})
print("\n--- Фильтр: category=operations ---")
for r in ops_only[:3]:
    print(f"  score={r['score']:.4f} | {r['metadata']['category']}/{r['metadata']['filename']}")

# Только concepts (теоретические статьи)
con_only = search_dense_filtered(query, {"category": "concepts"})
print("\n--- Фильтр: category=concepts ---")
for r in con_only[:3]:
    print(f"  score={r['score']:.4f} | {r['metadata']['category']}/{r['metadata']['filename']}")

print("\n💡 Фильтр по category=operations даёт только инструкции, по concepts — только теорию.")

🔎 Запрос: 'как создать чарт'

--- Без фильтра ---
  score=0.8829 | operations/create-chart
  score=0.8736 | operations/add-hierarchy
  score=0.8709 | operations/create-chart

--- Фильтр: category=operations ---
  score=0.8829 | operations/create-chart
  score=0.8736 | operations/add-hierarchy
  score=0.8709 | operations/create-chart

--- Фильтр: category=concepts ---
  score=0.8531 | concepts/dataset-based-charts
  score=0.8518 | concepts/dataset-based-charts
  score=0.8463 | concepts/index

💡 Фильтр по category=operations даёт только инструкции, по concepts — только теорию.


## Batch Processing

ChromaDB нативно поддерживает batch-запросы: `collection.query(query_texts=[q1, q2, ...])` выполняет
все эмбеддинги и HNSW-поиски за **один вызов** вместо N отдельных.

| Подход | Описание |
|---|---|
| Sequential | N вызовов `search_dense(q)` в цикле |
| Dense Batch | Один вызов `collection.query(query_texts=[...N запросов...])` |
| Hybrid Batch | Batch Dense + параллельный BM25 через ThreadPoolExecutor, затем RRF |

Метрика сравнения: **throughput (запросов/сек)** и **latency на весь батч**.

In [19]:
from concurrent.futures import ThreadPoolExecutor
import statistics


def search_dense_batch(queries: List[str], k: int = TOP_K) -> List[List[Dict]]:
    """ChromaDB нативный batch: один вызов для N запросов."""
    results = collection.query(
        query_texts=queries,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    return [
        [
            {
                "chunk_id": results["ids"][q_idx][i],
                "text": results["documents"][q_idx][i],
                "metadata": results["metadatas"][q_idx][i],
                "score": 1 - results["distances"][q_idx][i],
            }
            for i in range(len(results["ids"][q_idx]))
        ]
        for q_idx in range(len(queries))
    ]


def search_hybrid_batch(queries: List[str], k: int = TOP_K) -> List[List[Dict]]:
    """Hybrid batch: Dense batch + параллельный BM25, затем RRF."""
    dense_batch = search_dense_batch(queries, k=k)
    with ThreadPoolExecutor() as pool:
        sparse_batch = list(pool.map(lambda q: search_bm25(q, k=k), queries))
    return [
        reciprocal_rank_fusion(d, s)[:k]
        for d, s in zip(dense_batch, sparse_batch)
    ]


In [20]:
BATCH_SIZES = [1, 5, 10, 20]
RUNS = 10
base_queries = BENCH_QUERIES * 4  # 20 запросов (повтор для разных batch_size)

rows = []
for bs in BATCH_SIZES:
    batch = base_queries[:bs]

    seq_times, bat_times, hyb_times = [], [], []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        for q in batch:
            search_dense(q)
        seq_times.append((time.perf_counter() - t0) * 1000)

        t0 = time.perf_counter()
        search_dense_batch(batch)
        bat_times.append((time.perf_counter() - t0) * 1000)

        t0 = time.perf_counter()
        search_hybrid_batch(batch)
        hyb_times.append((time.perf_counter() - t0) * 1000)

    seq_ms  = statistics.mean(seq_times)
    bat_ms  = statistics.mean(bat_times)
    hyb_ms  = statistics.mean(hyb_times)

    rows.append({
        "batch_size": bs,
        "seq_total_ms": round(seq_ms,  1),
        "batch_total_ms": round(bat_ms,  1),
        "hybrid_total_ms": round(hyb_ms,  1),
        "speedup": round(seq_ms / bat_ms, 2),
        "seq_qps": round(bs / seq_ms * 1000, 1),
        "batch_qps": round(bs / bat_ms * 1000, 1),
    })

df_batch = pd.DataFrame(rows)

print("Latency на весь батч (ms):")
display(df_batch[["batch_size", "seq_total_ms", "batch_total_ms", "hybrid_total_ms", "speedup"]])

print("\nThroughput (запросов/сек):")
display(df_batch[["batch_size", "seq_qps", "batch_qps"]])

print("\nВыводы:")
for row in rows:
    bs = row["batch_size"]
    speedup = row["speedup"]
    sq = row["seq_qps"]
    bq = row["batch_qps"]
    delta = (bq / sq - 1) * 100
    arrow = "▲" if delta > 0 else "▼"
    print(f"  batch={bs:>2}: speedup={speedup:.2f}x | {sq:.0f} → {bq:.0f} qps ({arrow}{abs(delta):.0f}%)")

print()
print("speedup  = seq_total_ms / batch_total_ms")
print("ChromaDB batch кодирует все запросы одним вызовом encode(),")
print("экономя overhead на инициализацию и Python-циклы.")
print("Hybrid batch дополнительно распараллеливает BM25 через ThreadPoolExecutor.")

Latency на весь батч (ms):


,batch_size,seq_total_ms,batch_total_ms,hybrid_total_ms,speedup
0,1,24.6,18.7,19.0,1.31
1,5,109.9,25.9,26.3,4.25
2,10,214.4,36.0,37.2,5.95
3,20,413.7,56.2,58.7,7.36



Throughput (запросов/сек):


,batch_size,seq_qps,batch_qps
0,1,40.6,53.4
1,5,45.5,193.4
2,10,46.6,277.6
3,20,48.3,355.7



Выводы:
  batch= 1: speedup=1.31x | 41 → 53 qps (▲32%)
  batch= 5: speedup=4.25x | 46 → 193 qps (▲325%)
  batch=10: speedup=5.95x | 47 → 278 qps (▲496%)
  batch=20: speedup=7.36x | 48 → 356 qps (▲636%)

speedup  = seq_total_ms / batch_total_ms
ChromaDB batch кодирует все запросы одним вызовом encode(),
экономя overhead на инициализацию и Python-циклы.
Hybrid batch дополнительно распараллеливает BM25 через ThreadPoolExecutor.


### Анализ результатов Batch Processing

Sequential: каждый запрос — отдельный вызов `encode()` + HNSW-поиск. Latency = N × ~21ms → линейный рост.

Batch: один вызов `encode([q1..qN])` — SentenceTransformer обрабатывает все запросы за один forward pass по модели. Overhead на вызов платится один раз, N запросов «параллелизуются» внутри матричного умножения.

**Throughput batch масштабируется почти линейно:**

```
batch= 1:  53 qps  (baseline)
batch= 5: 193 qps  (+264%)
batch=10: 278 qps  (+424%)
batch=20: 356 qps  (+571%)
```

Sequential throughput почти не растёт (~40–48 qps) — он ограничен накладными расходами на каждый вызов.

**Hybrid ≈ Dense batch по latency:**
Добавление BM25 через `ThreadPoolExecutor` даёт +1–3ms на батч — практически бесплатно, поскольку BM25 занимает <0.1ms на запрос и выполняется параллельно с encode().

При batch=20 один вызов заменяет 20 последовательных и даёт 7.4x ускорение по latency и 7.4x прирост throughput (48 → 356 qps).